In [1]:
# 1. Import libraries
import pandas as pd
import re
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# 2. Load dataset
df = pd.read_csv("alias_list.csv")
print(df.head())
print(df.columns)

# 3. Select alias column
# Change "alias" if your column has a different name
df = df.dropna(subset=["alias"])
df["clean"] = df["alias"].astype(str).str.lower()
df["clean"] = df["clean"].apply(
    lambda x: re.sub(r"[^a-zA-Z0-9\s]", "", x)
)

# 4. NLP: TF-IDF
tfidf = TfidfVectorizer(stop_words="english")
X = tfidf.fit_transform(df["clean"])

# 5. K-Means
k = 5
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X)

# 6. Show clusters
print(df[["alias", "cluster"]].sort_values("cluster"))

# 7. Important words in each cluster
terms = tfidf.get_feature_names_out()

for i, center in enumerate(kmeans.cluster_centers_):
    words = [terms[j] for j in center.argsort()[-10:][::-1]]
    print(f"Cluster {i}: {', '.join(words)}")

# 8. Visualise clusters
pca = PCA(n_components=2, random_state=42)
points = pca.fit_transform(X.toarray())

plt.figure(figsize=(8, 5))
plt.scatter(points[:, 0], points[:, 1], c=df["cluster"])
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Alias K-Means Clustering")
plt.show()

# 9. Save results
df.to_csv("alias_list_clustered.csv", index=False)
print("Done! Saved as alias_list_clustered.csv")

       Character                                            Aliases
0    Aaron Davis                                Aaron Davis,Prowler
1    Abomination  Abomination, The Abomination, Emil Blonsky, Bl...
2     Abu Bakaar                                         Abu Bakaar
3       Agent 13           Agent 13, Kate / Agent 13, Sharon Carter
4  Agent Garrett             Agent Garrett, Jonathan 'John' Garrett
Index(['Character', 'Aliases'], dtype='str')


KeyError: ['alias']